In [6]:
import sys, glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/Projet Multimédia/Partie audio/ghislain")
import utils

PREDS = Path("/content/drive/MyDrive/Projet Multimédia/Partie audio/ghislain/preds")
F_ANCIEN = PREDS / "audio_hubert_dev_test.csv"       # le fichier actuel (couche 2 ?)
E = sorted(utils.EMOTIONS)                            # ordre alphabétique, comme le binôme

# ------------------------------------------------------------------
# 1. Trouver hubert_base.npy et ids.txt sur le Drive
#    (dossier partagé par le binôme : ajoutez un raccourci "projet_emovid" dans Mon Drive si besoin)
# ------------------------------------------------------------------
def trouver(nom):
    candidats = [f"/content/drive/MyDrive/projet_emovid/embeddings/{nom}"] + \
                glob.glob(f"/content/drive/**/{nom}", recursive=True)
    for c in candidats:
        if Path(c).exists():
            return c
    raise FileNotFoundError(f"{nom} introuvable : ajoutez le dossier projet_emovid du binôme à Mon Drive")

F_NPY, F_IDS = trouver("hubert_base.npy"), trouver("ids.txt")
print("Embeddings :", F_NPY, "\nIdentifiants :", F_IDS)

X = np.load(F_NPY)                                    # (3088, 13, 768)
ids = [l.strip().replace(".mp3", "").replace(".wav", "") for l in open(F_IDS) if l.strip()]
assert len(ids) == len(X), f"{len(ids)} ids pour {len(X)} vecteurs"
print("Forme :", X.shape)

# ------------------------------------------------------------------
# 2. Partition officielle et vraies émotions
# ------------------------------------------------------------------
P = utils.setup(parts=("annotation",))
meta = utils.load_metadata(P).set_index("video_id")
split = meta.loc[ids, "split"].to_numpy()
y = meta.loc[ids, "emotion"].to_numpy()               # noms d'émotions -> classes alphabétiques
tr, dv, te = (split == "train"), (split == "dev"), (split == "test")
print("train / dev / test :", tr.sum(), dv.sum(), te.sum())

def probas(Xc, C=0.0005):
    """Pipeline du binôme : standardisation sur le train + régression logistique équilibrée."""
    sc = StandardScaler().fit(Xc[tr])
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, random_state=42)
    clf.fit(sc.transform(Xc[tr]), y[tr])
    assert list(clf.classes_) == E
    return clf.predict_proba(sc.transform(Xc))

f1 = lambda m, p: f1_score(y[m], np.array(E)[p[m].argmax(1)], average="macro")

# ------------------------------------------------------------------
# 3. Quelle configuration a produit l'ancien fichier ?
# ------------------------------------------------------------------
ancien = pd.read_csv(F_ANCIEN, dtype={"video_id": str}).set_index("video_id")
ancien = ancien.rename(columns=lambda c: c.replace("proba_", ""))
masque = dv | te
ids_dt = np.array(ids)[masque]
P_ancien = ancien.loc[ids_dt, E].to_numpy()

configs = {f"couche {k}": X[:, k, :] for k in range(13)}
configs["moyenne 13 couches"] = X.mean(axis=1)

lignes = []
for nom, Xc in configs.items():
    p = probas(Xc)
    lignes.append({"configuration": nom, "F1 dev": f1(dv, p), "F1 test": f1(te, p),
                   "écart max avec l'ancien fichier": np.abs(p[masque] - P_ancien).max()})
    print(f"{nom:20s} dev {lignes[-1]['F1 dev']:.3f} | test {lignes[-1]['F1 test']:.3f} "
          f"| écart avec l'ancien fichier {lignes[-1]['écart max avec l\'ancien fichier']:.2e}")
tab = pd.DataFrame(lignes).set_index("configuration")
origine = tab["écart max avec l'ancien fichier"].idxmin()
print(f"\n=> L'ancien fichier correspond à : {origine} "
      f"(écart {tab.loc[origine].iloc[2]:.1e} ; ~0 = identique)")

# ------------------------------------------------------------------
# 4. Bonnes probabilités : moyenne des 13 couches, C = 0.0005
# ------------------------------------------------------------------
p_bon = probas(configs["moyenne 13 couches"])
print(f"\nConfiguration retenue : dev {f1(dv, p_bon):.3f} | test {f1(te, p_bon):.3f} "
      "(attendu : 0.274 / 0.214)")

sortie = pd.DataFrame({"video_id": ids_dt, "split": split[masque], "emotion": y[masque]})
sortie[E] = p_bon[masque]
chemin = PREDS / "audio_hubert_corrige.csv"
sortie.to_csv(chemin, index=False)
print("Fichier enregistré :", chemin, "|", sortie.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Embeddings : /content/drive/MyDrive/Projet Multimédia/Partie audio/MES PROBA/embeddings/hubert_base.npy 
Identifiants : /content/drive/MyDrive/Projet Multimédia/Partie audio/MES PROBA/embeddings/ids.txt
Forme : (3088, 13, 768)
Données  : /content/data/PROJET_SID_2026
Résultats: /content/drive/MyDrive/Projet Multimédia/Partie audio/ghislain
train / dev / test : 2207 442 439
couche 0             dev 0.229 | test 0.201 | écart avec l'ancien fichier 2.27e-01
couche 1             dev 0.231 | test 0.202 | écart avec l'ancien fichier 2.02e-01
couche 2             dev 0.238 | test 0.182 | écart avec l'ancien fichier 1.55e-14
couche 3             dev 0.235 | test 0.196 | écart avec l'ancien fichier 1.30e-01
couche 4             dev 0.256 | test 0.221 | écart avec l'ancien fichier 1.63e-01
couche 5             dev 0.262 | test 0.210 | écart avec l'ancien fichier 1.83

### Comment fonctionne ce code : le déroulement pas à pas

**L'idée générale.** Nous avons trois modèles qui donnent chacun leur avis sur l'émotion d'un extrait de film. HuBERT écoute la voix, AST écoute les sons (musique, cris, bruits) et le modèle vidéo regarde les images. Chacun donne, pour chaque extrait, 8 probabilités : par exemple « 40 % peur, 20 % colère, 10 % tristesse… ». Ces avis ne sont pas toujours d'accord. Le but du code est d'apprendre à **combiner ces trois avis** pour obtenir une meilleure décision que chaque modèle seul. Pour cela, on utilise un **réseau de neurones**, un programme qui apprend à partir d'exemples.

**Étape 1 : rassembler les avis des trois modèles.** Le code lit les fichiers de probabilités des trois modèles, pour la partie dev (442 extraits) et pour la partie test (439 extraits). Il vérifie que les trois fichiers parlent bien des mêmes extraits et que les émotions sont rangées dans le même ordre, pour ne pas mélanger, par exemple, la « peur » d'un modèle avec la « colère » d'un autre. Pour chaque extrait, les 8 probabilités de chaque modèle sont mises bout à bout, ce qui donne une liste de **24 nombres** : c'est ce que le réseau recevra en entrée. On prend aussi le logarithme de ces nombres et on les remet à la même échelle, ce qui facilite l'apprentissage.

**Étape 2 : préparer des points de comparaison.** Avant d'entraîner le réseau, le code calcule des méthodes plus simples, pour savoir ensuite si le réseau fait vraiment mieux :
- **chaque modèle seul**, sans combinaison ;
- **une moyenne pondérée**, où l'on donne à chaque modèle un « poids », par exemple 80 % pour la vidéo et 20 % pour HuBERT, puis on fait la moyenne de leurs avis. Les poids sont choisis sur le dev ;
- **une régression logistique**, une méthode statistique classique, plus simple qu'un réseau de neurones, qui apprend elle aussi à combiner les 24 nombres.

**Étape 3 : construire le réseau de neurones.** Le réseau est volontairement petit, car nous n'avons que 442 extraits pour lui apprendre. Il reçoit les 24 nombres, les fait passer par une couche intermédiaire de 16 ou 32 « neurones », qui cherchent des combinaisons utiles (par exemple « la voix ET l'image disent toutes les deux peur »), puis il donne 8 probabilités finales, une par émotion. Deux protections l'empêchent d'apprendre les exemples par cœur au lieu de règles générales. Le **dropout** éteint au hasard certains neurones pendant l'apprentissage. Le **weight decay** l'empêche de donner trop d'importance à un seul détail. Enfin, les émotions rares comptent autant que les fréquentes pendant l'apprentissage.

**Étape 4 : choisir les bons réglages, sans tricher.** Le réseau a des réglages qu'on ne connaît pas à l'avance : nombre de neurones, force des protections, durée de l'apprentissage. Pour les choisir, on découpe les 442 extraits du dev en **5 groupes de films**. On entraîne le réseau sur 4 groupes et on regarde son score sur le 5ᵉ groupe, qu'il n'a jamais vu, puis on recommence en changeant de groupe, 5 fois au total. C'est ce qu'on appelle la **validation croisée**. Les groupes sont faits **par film**, pour que deux extraits d'un même film (mêmes acteurs, même musique) ne soient jamais l'un en apprentissage et l'autre en évaluation, ce qui rendrait la tâche trop facile. On garde les réglages qui donnent le meilleur score moyen.

**Étape 5 : entraîner le réseau final.** Avec les meilleurs réglages, le réseau est entraîné sur **tous les extraits du dev**. Comme un petit réseau entraîné sur peu de données peut donner des résultats un peu différents d'un essai à l'autre, on l'entraîne **5 fois** avec des départs aléatoires différents, et on fait **la moyenne** de leurs prédictions. Le résultat est ainsi plus stable.

**Étape 6 : évaluer sur le test.** Le réseau est enfin appliqué **une seule fois** aux 439 extraits du test, qu'il n'a jamais vus et qui n'ont servi à aucun réglage. C'est la mesure honnête de ses performances. Le score utilisé est le **F1 macro**, qui donne le même poids aux 8 émotions, qu'elles soient rares ou fréquentes. Le code affiche un tableau qui compare le réseau aux méthodes simples de l'étape 2, donne le détail par émotion et selon la présence ou non de parole, puis enregistre les prédictions du réseau dans le fichier `stacking_mlp_test.csv`.

**En résumé :** le réseau apprend sur le dev à combiner les avis de la voix, des sons et des images, ses réglages sont choisis sans jamais toucher au test, et on vérifie à la fin, sur le test, s'il fait mieux que chaque modèle seul et que des combinaisons plus simples.

In [7]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
import torch
import torch.nn as nn

from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/Projet Multimédia/Partie audio/ghislain")
import utils

# ------------------------------------------------------------------
# 0. Paramètres : adapter les chemins des fichiers vidéo si besoin
# ------------------------------------------------------------------
PREDS = Path("/content/drive/MyDrive/Projet Multimédia/Partie audio/ghislain/preds")
SOURCES = {
    "HuBERT": {"dev": PREDS / "audio_hubert_corrige.csv", "test": PREDS / "audio_hubert_corrige.csv"},
    "AST":    {"dev": PREDS / "ast_dev.csv",              "test": PREDS / "ast_test.csv"},
    "vidéo":  {"dev": PREDS / "probas_video_dev.csv",     "test": PREDS / "probas_video_test.csv"},
}
EMOTIONS = utils.EMOTIONS
SEED = 42

P = utils.setup(parts=("annotation",))
meta = utils.load_metadata(P).set_index("video_id")

# ------------------------------------------------------------------
# 1. Lecture : extraits choisis selon la partition officielle, émotions alignées par nom
# ------------------------------------------------------------------
def lire(fichier, split):
    df = pd.read_csv(fichier, dtype={"video_id": str})
    df["video_id"] = df["video_id"].str.strip().str.replace(r"\.(wav|mp3|mp4)$", "", regex=True)
    df = df.rename(columns=lambda c: c.replace("proba_", ""))
    manquantes = [e for e in EMOTIONS if e not in df.columns]
    assert not manquantes, f"{fichier.name} : colonnes manquantes {manquantes}"
    df = df.drop_duplicates("video_id").set_index("video_id")
    return df[df.index.isin(meta.index[meta["split"] == split])][EMOTIONS]

donnees = {}
for split in ("dev", "test"):
    tables = {nom: lire(f[split], split) for nom, f in SOURCES.items()}
    ids = sorted(set.intersection(*(set(t.index) for t in tables.values())))
    print(f"{split} : " + " | ".join(f"{n} {len(t)}" for n, t in tables.items()) + f" | communs {len(ids)}")
    donnees[split] = {
        "ids": ids,
        "probas": {n: t.loc[ids].to_numpy() for n, t in tables.items()},
        "y": meta.loc[ids, "label"].to_numpy(),
        "films": meta.loc[ids, "film_id"].to_numpy(),
    }
assert len(donnees["dev"]["ids"]) == 442 and len(donnees["test"]["ids"]) == 439

def entrees(split):
    """24 valeurs par extrait : log des probabilités des 3 sources."""
    return np.hstack([np.log(np.clip(p, 1e-6, 1)) for p in donnees[split]["probas"].values()])

X_dev, X_test = entrees("dev"), entrees("test")
y_dev, y_test = donnees["dev"]["y"], donnees["test"]["y"]
f1 = lambda y, p: f1_score(y, p.argmax(1), average="macro")

# Découpage du dev en 5 groupes de films (aucun film à la fois en apprentissage et en validation)
plis = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
            .split(X_dev, y_dev, donnees["dev"]["films"]))

# ------------------------------------------------------------------
# 2. Références
# ------------------------------------------------------------------
resultats = []
for nom in SOURCES:                                   # 2a. chaque source seule
    resultats.append({"méthode": f"{nom} seul", "F1 dev": f1(y_dev, donnees["dev"]["probas"][nom]),
                      "F1 test": f1(y_test, donnees["test"]["probas"][nom])})

# 2b. Moyenne pondérée : poids (somme = 1, pas de 0,1) choisis sur le dev
noms = list(SOURCES)
grille = [w for w in product(np.round(np.arange(0, 1.01, 0.1), 1), repeat=len(noms)) if abs(sum(w) - 1) < 1e-9]
moyenne = lambda split, w: sum(wi * donnees[split]["probas"][n] for wi, n in zip(w, noms))
w_best = max(grille, key=lambda w: f1(y_dev, moyenne("dev", w)))
resultats.append({"méthode": "moyenne pondérée " + str(dict(zip(noms, w_best))),
                  "F1 dev": f1(y_dev, moyenne("dev", w_best)), "F1 test": f1(y_test, moyenne("test", w_best))})

# 2c. Régression logistique sur les 24 valeurs (C choisi par validation croisée sur le dev)
def cv_logreg(C):
    p = np.zeros((len(y_dev), len(EMOTIONS)))
    for a, v in plis:
        sc = StandardScaler().fit(X_dev[a])
        m = LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit(sc.transform(X_dev[a]), y_dev[a])
        p[v] = m.predict_proba(sc.transform(X_dev[v]))
    return f1(y_dev, p)

C_best = max([0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0], key=cv_logreg)
sc = StandardScaler().fit(X_dev)
lr = LogisticRegression(C=C_best, class_weight="balanced", max_iter=5000).fit(sc.transform(X_dev), y_dev)
resultats.append({"méthode": f"régression logistique (C={C_best})", "F1 dev": cv_logreg(C_best),
                  "F1 test": f1(y_test, lr.predict_proba(sc.transform(X_test)))})

# ------------------------------------------------------------------
# 3. Réseau de neurones : 24 -> cachée -> 8
# ------------------------------------------------------------------
torch.manual_seed(SEED)

class Reseau(nn.Module):
    def __init__(self, cachee, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(X_dev.shape[1], cachee), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(cachee, len(EMOTIONS)))
    def forward(self, x):
        return self.net(x)

def entrainer(Xa, ya, cachee, dropout, wd, epoques, graine, X_suivi=None, y_suivi=None):
    """Entraîne le réseau ; si X_suivi est fourni, renvoie le F1 de validation toutes les 10 époques."""
    torch.manual_seed(graine)
    sc = StandardScaler().fit(Xa)
    xa = torch.tensor(sc.transform(Xa), dtype=torch.float32)
    ya_t = torch.tensor(ya)
    poids = torch.tensor(len(ya) / (len(EMOTIONS) * np.bincount(ya, minlength=len(EMOTIONS)).clip(1)),
                         dtype=torch.float32)                         # classes équilibrées
    modele = Reseau(cachee, dropout)
    opt = torch.optim.AdamW(modele.parameters(), lr=1e-2, weight_decay=wd)
    perte = nn.CrossEntropyLoss(weight=poids)
    suivi = []
    for ep in range(1, epoques + 1):
        modele.train()
        opt.zero_grad()
        perte(modele(xa), ya_t).backward()
        opt.step()
        if X_suivi is not None and ep % 10 == 0:
            modele.eval()
            with torch.no_grad():
                pv = modele(torch.tensor(sc.transform(X_suivi), dtype=torch.float32)).argmax(1).numpy()
            suivi.append(f1_score(y_suivi, pv, average="macro"))
    modele.eval()
    predire = lambda X: torch.softmax(modele(torch.tensor(sc.transform(X), dtype=torch.float32)), 1).detach().numpy()
    return predire, suivi

# 3a. Choix des réglages par validation croisée sur le dev
EPOQUES_MAX = 300
meilleur = None
for cachee, dropout, wd in product((16, 32), (0.3, 0.5), (1e-3, 1e-2, 1e-1)):
    courbes = np.array([entrainer(X_dev[a], y_dev[a], cachee, dropout, wd, EPOQUES_MAX, SEED,
                                  X_dev[v], y_dev[v])[1] for a, v in plis])
    moy = courbes.mean(0)                              # F1 moyen des 5 plis, toutes les 10 époques
    k = int(moy.argmax())
    print(f"cachée {cachee:2d} | dropout {dropout} | wd {wd:<5} -> F1 CV {moy[k]:.3f} à {10 * (k + 1)} époques")
    if meilleur is None or moy[k] > meilleur["F1"]:
        meilleur = {"cachee": cachee, "dropout": dropout, "wd": wd, "epoques": 10 * (k + 1), "F1": moy[k]}
print("\nRéglages retenus :", meilleur)

# 3b. Entraînement final sur tout le dev, moyenne de 5 graines, puis test (une seule fois)
P_mlp = np.mean([entrainer(X_dev, y_dev, meilleur["cachee"], meilleur["dropout"], meilleur["wd"],
                           meilleur["epoques"], graine)[0](X_test) for graine in range(5)], axis=0)
resultats.append({"méthode": "réseau de neurones (5 graines)", "F1 dev": meilleur["F1"],
                  "F1 test": f1(y_test, P_mlp)})

# ------------------------------------------------------------------
# 4. Comparaison et sauvegarde
# ------------------------------------------------------------------
tab = pd.DataFrame(resultats).set_index("méthode").round(3)
print("\nF1 dev : score sur le dev (validation croisée pour la régression logistique et le réseau)")
display(tab)

res = utils.evaluate(y_test, P_mlp, parole=meta.loc[donnees["test"]["ids"], "parole"].to_numpy(),
                     name="stacking_mlp_hubert_ast_video", split="test")
utils.log_results(res, P)

sortie = pd.DataFrame({"video_id": donnees["test"]["ids"],
                       "emotion_reelle": meta.loc[donnees["test"]["ids"], "emotion"].to_numpy(),
                       "emotion_predite": np.array(EMOTIONS)[P_mlp.argmax(1)]})
for i, e in enumerate(EMOTIONS):
    sortie[f"proba_{e}"] = P_mlp[:, i]
sortie.to_csv(PREDS / "stacking_mlp_test.csv", index=False)
print("Fichier enregistré :", PREDS / "stacking_mlp_test.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Données  : /content/data/PROJET_SID_2026
Résultats: /content/drive/MyDrive/Projet Multimédia/Partie audio/ghislain
dev : HuBERT 442 | AST 442 | vidéo 442 | communs 442
test : HuBERT 439 | AST 439 | vidéo 439 | communs 439
cachée 16 | dropout 0.3 | wd 0.001 -> F1 CV 0.273 à 50 époques
cachée 16 | dropout 0.3 | wd 0.01  -> F1 CV 0.273 à 50 époques
cachée 16 | dropout 0.3 | wd 0.1   -> F1 CV 0.273 à 50 époques
cachée 16 | dropout 0.5 | wd 0.001 -> F1 CV 0.263 à 50 époques
cachée 16 | dropout 0.5 | wd 0.01  -> F1 CV 0.261 à 50 époques
cachée 16 | dropout 0.5 | wd 0.1   -> F1 CV 0.264 à 50 époques
cachée 32 | dropout 0.3 | wd 0.001 -> F1 CV 0.272 à 20 époques
cachée 32 | dropout 0.3 | wd 0.01  -> F1 CV 0.272 à 20 époques
cachée 32 | dropout 0.3 | wd 0.1   -> F1 CV 0.272 à 20 époques
cachée 32 | dropout 0.5 | wd 0.001 -> F1 CV 0.272 à 30 époques
cachée 32 | dropout

,F1 dev,F1 test
méthode,,
HuBERT seul,0.274,0.214
AST seul,0.236,0.248
vidéo seul,0.406,0.351
"moyenne pondérée {'HuBERT': np.float64(0.2), 'AST': np.float64(0.0), 'vidéo': np.float64(0.8)}",0.414,0.351
régression logistique (C=0.03),0.331,0.320
réseau de neurones (5 graines),0.273,0.335


[stacking_mlp_hubert_ast_video | test] F1 macro = 0.335 | acc. équilibrée = 0.342 | accuracy = 0.351
   F1 macro avec parole = 0.309 | sans parole = 0.320
   F1 par émotion : amusement 0.39, awe 0.31, contentment 0.37, excitement 0.23, anger 0.43, disgust 0.25, fear 0.44, sadness 0.26
Fichier enregistré : /content/drive/MyDrive/Projet Multimédia/Partie audio/ghislain/preds/stacking_mlp_test.csv


## Fusion audio + vidéo par réseau de neurones (stacking) : méthode et conclusion

### Méthode

Nous avons combiné les probabilités de trois modèles : HuBERT (audio, parole), AST (audio, sons généraux) et le modèle vidéo. Pour chaque extrait, les 8 probabilités de chaque modèle sont mises bout à bout, ce qui donne 24 valeurs. On prend leur logarithme, puis on les standardise. Ces 24 valeurs sont les entrées d'un petit réseau de neurones à une couche cachée (16 ou 32 neurones, activation ReLU, dropout), avec 8 sorties, une par émotion. La perte est une entropie croisée pondérée par classe, pour que les émotions rares comptent autant que les fréquentes, ce qui est cohérent avec le F1 macro. L'optimiseur est AdamW, avec un weight decay qui limite le surapprentissage.

Le réseau est entraîné uniquement sur les probabilités du dev (442 extraits). Les réglages (taille de la couche cachée, dropout, weight decay, nombre d'époques) sont choisis par validation croisée en 5 plis à l'intérieur du dev. Les plis sont découpés par film, pour qu'aucun film ne serve à la fois à l'apprentissage et à l'évaluation. Le réseau final est entraîné sur tout le dev avec 5 graines aléatoires, et on fait la moyenne de leurs prédictions pour stabiliser le résultat. Il est ensuite appliqué une seule fois au test (439 extraits). Nous le comparons à trois références : chaque modèle seul, une moyenne pondérée des probabilités (poids choisis sur le dev) et une régression logistique sur les mêmes 24 valeurs (C choisi par la même validation croisée).

### Résultats (F1 macro)

| Méthode | Dev | Test |
|---|---|---|
| HuBERT seul | 0,274 | 0,214 |
| AST seul | 0,236 | 0,248 |
| Vidéo seule | 0,406 | **0,351** |
| Moyenne pondérée (vidéo 0,8 / HuBERT 0,2 / AST 0) | 0,414 | **0,351** |
| Régression logistique (C = 0,03) | 0,331* | 0,320 |
| Réseau de neurones (5 graines) | 0,273* | 0,335 |

\* Score de validation croisée sur le dev.

### Conclusion

La vidéo est de loin la modalité la plus informative : seule, elle atteint 0,351 au test, contre 0,21 à 0,25 pour les modèles audio. Aucune méthode de fusion ne fait mieux qu'elle. La moyenne pondérée donne presque tout le poids à la vidéo et obtient exactement le même score au test, ce qui montre que son léger gain sur le dev venait du réglage sur le dev lui-même. La régression logistique (0,320) et le réseau de neurones (0,335) font même un peu moins bien que la vidéo seule. Avec seulement 442 extraits pour apprendre à combiner 24 valeurs, soit environ 55 par émotion, ces modèles perdent une partie de la qualité de la vidéo en cherchant à exploiter l'audio, qui est faible.

Ces écarts restent cependant petits par rapport à l'incertitude de la mesure : sur 439 extraits, l'intervalle de confiance d'un F1 macro est d'environ ±0,05. On ne peut donc pas affirmer qu'une méthode est réellement meilleure qu'une autre. On peut seulement dire que la fusion avec l'audio n'apporte pas de gain mesurable. Le réseau reconnaît le mieux la peur (0,44) et la colère (0,43), qui sont les émotions qui s'entendent le plus (cris, voix tendue, musique), et le moins bien l'excitation (0,23), le dégoût (0,25) et la tristesse (0,26). Son score est semblable avec parole (0,309) et sans parole (0,320), signe qu'il s'appuie surtout sur la vidéo.

En résumé, sur ce corpus, la vidéo seule est la meilleure solution, à égalité avec la moyenne pondérée. Les modèles audio sont trop faibles, et le dev trop petit, pour qu'une fusion apprise en tire profit sans surapprendre. La configuration AST + caption + couleurs n'a pas été utilisée ici, car elle mélange le texte et des informations visuelles : elle relève de la fusion audio + vidéo + texte, traitée par un autre membre du groupe.